# 04 · Export GGUF quantizations

Merge the adapter to fp16, build llama.cpp, compute an importance matrix, and emit Q4_K_M / Q5_K_M / Q8_0. Then plot quality-vs-size.

In [ ]:
# --- Bootstrap: works on Kaggle, Colab, or locally ---------------------------
import os, sys, subprocess

# If you pushed this project to GitHub, set REPO_URL so Kaggle can clone it.
REPO_URL = "https://github.com/Niikhi/peft-function-calling.git"  # <-- EDIT ME
REPO_DIR = "peft-function-calling"

if not os.path.exists("src") and not os.path.exists(os.path.join(REPO_DIR, "src")):
    subprocess.run(["git", "clone", REPO_URL], check=True)
if os.path.exists(os.path.join(REPO_DIR, "src")):
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())

In [ ]:
!pip -q install unsloth

In [ ]:
from src.config import load_config
from src.model import load_model_and_tokenizer
from src.gguf_convert import merge_adapter_unsloth, run_full_conversion
import json

cfg = load_config()
cfg.model["base_id"] = str(cfg.path("adapter_dir"))  # load base + trained adapter
model, tokenizer = load_model_and_tokenizer(cfg)
merged_dir = merge_adapter_unsloth(cfg, model, tokenizer)

### Build llama.cpp + convert + quantize
(Takes several minutes; needs internet to clone llama.cpp.)

In [ ]:
rows = run_full_conversion(cfg, tokenizer, merged_dir, workdir=str(cfg.root))
with open(cfg.file_in("metrics_dir", "gguf_manifest.json"), "w") as f:
    json.dump(rows, f, indent=2)
rows

### Quality-vs-size chart
Attach the fine-tuned exact-match (from notebook 03) to each quant as a quality proxy; for true per-quant quality, serve each GGUF via Ollama/llama-cli and re-run `run_eval`.

In [ ]:
import json, os
from src.visualize import plot_quant_quality_size
from IPython.display import Image

ft_path = cfg.file_in("metrics_dir", "eval_finetuned.json")
ft_exact = json.load(open(ft_path))["summary"]["exact_match"] if os.path.exists(ft_path) else None
for r in rows:
    r.setdefault("exact_match", ft_exact or 0.0)

Image(plot_quant_quality_size(rows, cfg.file_in("figures_dir", "quant_quality_size.png")))